# Error Models Building

---

This notebook re-trains the UNIQUE error models (regressors predicting the absolute error `l1_error` of each Chemprop property model) for the public ADME endpoints and benchmarks them along three axes:

- **Input dataset benchmarking**: vary the amount of training / calibration data used to fit the error model.
- **ML algorithm benchmarking**: compare Random Forest vs. Lasso as error-model algorithm using the same UQ-metrics + predictions input.
- **Input features benchmarking**: compare error models trained on latent fingerprint features only vs. on the predicted value only, against the default UQ-metrics-based model.

The outputs of this notebook (`inputdata_benchmarking_data.csv`, `mlalgorithm_benchmarking_data.csv`, `inputfeatures_benchmarking_data.csv`) are consumed by `./Figures_ErrorModels_PublicData.ipynb` to reproduce the corresponding paper figures.

The notebook expects that the UNIQUE pipeline has already been executed for each endpoint (see `../../scripts/run_unique.py`) so that the corresponding `error_models/` subfolders exist under `../../unique_output/{model_id}/`.

In [ ]:
import pandas as pd
import numpy as np
import pickle
import warnings

from sklearn.ensemble import RandomForestRegressor

warnings.filterwarnings('ignore')

In [2]:
# Define the endpoints dictionary: {endpoint_group: {model_name: model_id}}
endpoints_dict = {'clint': {'rLM LogCLint': 'clint_rat', 'hLM LogCLint': 'clint_human'},
                  'ppb':   {'LogFu-Rat': 'ppb_rat', 'LogFu-Human': 'ppb_human'},
                  'permeability': {'MDCK-MDR1_LogER': 'mdck_mdr1_er'}}

## Input Dataset Benchmarking

Refit the default RF error model on varying subsets of the training data:
- 25 / 50 / 75 / 100 % of `TRAIN` (plus the full `CALIBRATION` set).
- 25 / 50 / 75 / 100 % of the `CALIBRATION` set only.

In [ ]:
dataset_sizes = []

for endpoint, models_dict in endpoints_dict.items():
    print(f'\n### {endpoint} ###\n')

    for model_name, model_id in models_dict.items():
        print(f'\n### {model_name} ###\n')
        # Load UNIQUE input data
        data = pd.read_csv(f'unique_input_data/{model_name}_unique_input_data.csv', index_col=0).reset_index(drop=True)
        print(data.shape)
        # Load error model input data (after standardization)
        em_input_data = pd.read_csv(f'unique_output/{model_id}/error_models/UQmetrics_predictions/data.csv', index_col=0).reset_index(drop=True)
        print(em_input_data.shape)
        # Drop columns and update column names
        em_input_data = em_input_data.drop(columns=['labels', 'predictions'], axis=1)
        em_input_data.columns = ['Manhattan Distance', 'Ensemble Variance', 'Predictions', 'l1_error']
        # Aggregate both dataframes
        df = pd.concat([data[['Id', 'which_set']], em_input_data], axis=1)
        print(df.shape)

        dataset_sizes.append([model_name,
                              'UniqueRandomForestRegressor[UQmetrics+predictions](l1)',
                              len(df.loc[df.which_set.isin(['TRAIN', 'CALIBRATION'])])])

        # Load the original error model and extract model hyperparameters
        with open(f'unique_output/{model_id}/error_models/UQmetrics_predictions/UniqueRandomForestRegressor_model.pkl', 'rb') as file:
            original_em = pickle.load(file)
        kwargs = original_em.get_params()

        # Select training data
        train_data = df[df['which_set'].isin(['TRAIN'])]

        for percent_mols in [25, 50, 75, 100]:
            # Select training data's subset
            selected_train_data = train_data.sample(frac=(percent_mols / 100), random_state=42)
            if percent_mols != 100:
                # Append calibration data
                selected_train_data = pd.concat([selected_train_data, df[df['which_set'].isin(['CALIBRATION'])]], ignore_index=True)
            print(selected_train_data.shape)
            # Select input features and target variable
            x_train = selected_train_data[['Manhattan Distance', 'Ensemble Variance', 'Predictions']]
            y_train = selected_train_data['l1_error']
            # Initialize and fit RF error model
            error_model = RandomForestRegressor(**kwargs)
            error_model.fit(x_train, y_train)
            # Predict on the entire dataset
            x_all = df[['Manhattan Distance', 'Ensemble Variance', 'Predictions']]
            df[f'em_predictions_{percent_mols}train'] = error_model.predict(x_all)

            dataset_sizes.append([model_name, f'em_predictions_{percent_mols}train', len(selected_train_data)])

        # Save the resulting dataframe
        df.to_csv(f'unique_output/{model_id}/error_models/UQmetrics_predictions/inputdata_benchmarking_data.csv', index=False)


### clint ###


### rLM LogCLint ###

(3051, 7)
(3051, 6)
(3051, 6)
(1297, 6)
(1679, 7)
(2060, 8)
(1527, 6)
(228, 10)
(457, 10)
(686, 10)
(915, 10)

### hLM LogCLint ###

(3084, 7)
(3084, 6)
(3084, 6)
(1310, 6)
(1696, 7)
(2082, 8)
(1544, 6)
(231, 10)
(462, 10)
(693, 10)
(924, 10)

### ppb ###


### LogFu-Rat ###

(167, 7)
(167, 6)
(167, 6)
(70, 6)
(92, 7)
(113, 8)
(87, 6)
(12, 10)
(24, 10)
(36, 10)
(48, 10)

### LogFu-Human ###

(193, 7)
(193, 6)
(193, 6)
(82, 6)
(106, 7)
(130, 8)
(96, 6)
(14, 10)
(29, 10)
(43, 10)
(58, 10)

### permeability ###


### MDCK-MDR1_LogER ###

(2639, 7)
(2639, 6)
(2639, 6)
(1121, 6)
(1451, 7)
(1782, 8)
(1321, 6)
(197, 10)
(395, 10)
(593, 10)
(791, 10)


In [4]:
endpoints_list = ['rLM LogCLint', 'hLM LogCLint', 'LogFu-Rat', 'LogFu-Human', 'MDCK-MDR1_LogER']
error_models_list = ['UniqueRandomForestRegressor[UQmetrics+predictions](l1)',
                     'em_predictions_100train', 'em_predictions_75train', 'em_predictions_50train', 'em_predictions_25train',
                     'em_predictions_100calib', 'em_predictions_75calib', 'em_predictions_50calib', 'em_predictions_25calib']

In [5]:
sizes_df = pd.DataFrame(dataset_sizes, columns=['Endpoint', 'EM', 'N'])
sizes_table = pd.pivot_table(sizes_df, values='N', index='Endpoint', columns='EM')
sizes_table = sizes_table.reindex(index=endpoints_list, columns=error_models_list)
sizes_table

EM,UniqueRandomForestRegressor[UQmetrics+predictions](l1),em_predictions_100train,em_predictions_75train,em_predictions_50train,em_predictions_25train,em_predictions_100calib,em_predictions_75calib,em_predictions_50calib,em_predictions_25calib
Endpoint,,,,,,,,,
rLM LogCLint,2442.0,1527.0,2060.0,1679.0,1297.0,915.0,686.0,457.0,228.0
hLM LogCLint,2468.0,1544.0,2082.0,1696.0,1310.0,924.0,693.0,462.0,231.0
LogFu-Rat,135.0,87.0,113.0,92.0,70.0,48.0,36.0,24.0,12.0
LogFu-Human,154.0,96.0,130.0,106.0,82.0,58.0,43.0,29.0,14.0
MDCK-MDR1_LogER,2112.0,1321.0,1782.0,1451.0,1121.0,791.0,593.0,395.0,197.0


## Input Features Benchmarking

Compare the default UNIQUE error model (trained on latent fingerprint + UQ metrics + predictions) against two alternatives:

- A Random Forest trained on the **300 latent fingerprint features only**.
- A Linear Regression trained on the **predicted value only**.

In [ ]:
for endpoint, models_dict in endpoints_dict.items():
    print(f'\n### {endpoint} ###\n')

    for model_name, model_id in models_dict.items():
        print(f'\n### {model_name} ###\n')
        # Load UNIQUE input data
        data = pd.read_csv(f'unique_input_data/{model_name}_unique_input_data.csv', index_col=0).reset_index(drop=True)
        print(data.shape)

        # Load error model input data (after standardization)
        em_input_data = pd.read_csv(f'unique_output/{model_id}/error_models/latent_fp_scaled_UQmetrics_predictions/data.csv', index_col=0).reset_index(drop=True)
        print(em_input_data.shape)
        # Drop columns
        em_input_data = em_input_data.drop(columns=['labels', 'predictions'], axis=1)
        # Update column names and drop additional columns
        input_feature_names = [f'fp_{i}' for i in range(300)] + ['Manhattan Distance', 'Ensemble Variance', 'Predictions']
        em_input_data.columns = [input_feature_names[i] if col.startswith('feature_') else col for i, col in enumerate(em_input_data.columns)]

        # Aggregate both dataframes
        df = pd.concat([data[['Id', 'which_set']], em_input_data], axis=1)
        print(df.shape)

        ### ONLY DATA FEATURES ERROR MODEL
        # Load the original error model and extract model hyperparameters
        with open(f'unique_output/{model_id}/error_models/latent_fp_scaled_UQmetrics_predictions/UniqueRandomForestRegressor_model.pkl', 'rb') as file:
            original_em = pickle.load(file)
        kwargs = original_em.get_params()

        # Select training data's input features and target variable
        train_data = df[df['which_set'].isin(['TRAIN', 'CALIBRATION'])]
        x_train = train_data[[col for col in train_data.columns if col.startswith('fp_')]]
        y_train = train_data['l1_error']
        print(x_train.shape)
        print(y_train.shape)

        # Initialize and fit RF error model
        error_model = RandomForestRegressor(**kwargs)
        error_model.fit(x_train, y_train)

        # Save the new error model
        with open(f'unique_output/{model_id}/error_models/latent_fp_scaled_UQmetrics_predictions/UniqueRandomForestRegressor_model_onlydatafeatures.pkl', 'wb') as file:
            pickle.dump(error_model, file)

        # Predict on the entire dataset
        x_all = df[[col for col in df.columns if col.startswith('fp_')]]
        df['em_predictions_onlydatafeatures'] = error_model.predict(x_all)
        print(df.shape)

        # Save the resulting dataframe
        df.to_csv(f'unique_output/{model_id}/error_models/latent_fp_scaled_UQmetrics_predictions/inputfeatures_benchmarking_data.csv', index=False)


### clint ###


### rLM LogCLint ###

(3051, 7)
(3051, 306)
(3051, 306)
(2442, 300)
(2442,)
(3051, 307)
(2442, 1)
(2442,)
(3051, 308)

### hLM LogCLint ###

(3084, 7)
(3084, 306)
(3084, 306)
(2468, 300)
(2468,)
(3084, 307)
(2468, 1)
(2468,)
(3084, 308)

### ppb ###


### LogFu-Rat ###

(167, 7)
(167, 306)
(167, 306)
(135, 300)
(135,)
(167, 307)
(135, 1)
(135,)
(167, 308)

### LogFu-Human ###

(193, 7)
(193, 306)
(193, 306)
(154, 300)
(154,)
(193, 307)
(154, 1)
(154,)
(193, 308)

### permeability ###


### MDCK-MDR1_LogER ###

(2639, 7)
(2639, 306)
(2639, 306)
(2112, 300)
(2112,)
(2639, 307)
(2112, 1)
(2112,)
(2639, 308)
